# 📍 K-Nearest Neighbors
**ISLP Chapters 2 & 4 · Data Pattern Recognition for the Rest of Us**

> The most intuitive ML model: to classify a new person, find their K most similar people in the training data and take a majority vote. No equations to fit, no coefficients — just memory and distance.

### What you'll learn
- KNN classification: how it works step by step
- Why feature scaling is non-negotiable
- Choosing K with cross-validation
- Visualizing decision boundaries
- The curse of dimensionality — why KNN struggles at high dimensions

### Dataset
**UCI Adult Income** — 48,842 census records. Predict whether a person earns >$50K/year based on age, education, hours worked, and capital gains. A real business question with interpretable features.

### Time: ~55 minutes

In [ ]:
# UCI Adult Income dataset — predict whether income exceeds $50K/year
# 48,842 records from the 1994 US Census Bureau
# Classic classification benchmark used in fairness and ML research

import pandas as pd
import numpy as np

cols = ['age','workclass','fnlwgt','education','education_num','marital_status',
        'occupation','relationship','race','sex','capital_gain','capital_loss',
        'hours_per_week','native_country','income']

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'

try:
    adult = pd.read_csv(url, header=None, names=cols,
                        na_values=' ?', skipinitialspace=True)
    print("Loaded from UCI directly")
except:
    # Fallback: reconstruct a representative sample synthetically
    np.random.seed(42)
    n = 5000
    adult = pd.DataFrame({
        'age':           np.random.randint(18, 75, n),
        'education_num': np.random.randint(1, 16, n),
        'hours_per_week':np.random.randint(10, 70, n),
        'capital_gain':  np.random.choice([0]*9 + [5000], n),
        'capital_loss':  np.random.choice([0]*19 + [1800], n),
        'sex':           np.random.choice(['Male','Female'], n, p=[0.67,0.33]),
        'workclass':     np.random.choice(['Private','Self-emp','Gov','Other'], n, p=[0.7,0.1,0.15,0.05]),
    })
    log_odds = (-3 + 0.04*adult['age'] + 0.15*adult['education_num']
                + 0.02*adult['hours_per_week']
                + 0.3*(adult['sex']=='Male').astype(float))
    adult['income'] = np.where(1/(1+np.exp(-log_odds)) > np.random.uniform(0,1,n),
                               '>50K', '<=50K')
    print("Using synthetic Adult-like data (network unavailable)")

# Clean and encode
adult = adult.dropna()
adult['income_binary'] = (adult['income'].str.strip().str.contains('>50K')).astype(int)

print(f"Shape: {adult.shape}")
print(f"Income >$50K rate: {adult['income_binary'].mean():.1%}")
adult[['age','education_num','hours_per_week','capital_gain','income']].head()


## 📐 Part 1 — How KNN Works

To classify a new person:
1. Compute distance from this person to every training observation
2. Find the K nearest neighbors
3. Predict the majority class among those K neighbors

For **regression**: predict the *mean* of the K neighbors' values instead.

The only hyperparameter is **K**:
- Small K (K=1) → very flexible, low bias, high variance, memorizes training data
- Large K → smooth boundary, high bias, low variance, may miss fine patterns

**Why this dataset?** Whether someone earns >$50K depends on age, education, and hours worked — exactly the kind of multidimensional similarity reasoning KNN models naturally.

In [ ]:
# Prepare numeric features for KNN
# KNN needs standardized numeric features — distance is meaningless otherwise

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Use numeric features only for clean KNN demonstration
features = ['age', 'education_num', 'hours_per_week', 'capital_gain', 'capital_loss']
X = adult[features].values
y = adult['income_binary'].values

# Standardize (critical for KNN — income in dollars would dominate age in years)
scaler = StandardScaler()
X_s = scaler.fit_transform(X)

X_tr, X_te, y_tr, y_te = train_test_split(X_s, y, test_size=0.25,
                                             random_state=42, stratify=y)

print(f"Training: {X_tr.shape[0]:,}  |  Test: {X_te.shape[0]:,}")
print(f"Features: {features}")
print("\nWhy standardize? Without scaling:")
raw_knn = KNeighborsClassifier(n_neighbors=5).fit(
    adult[features].values[:len(X_tr)], y_tr)
raw_acc = raw_knn.score(adult[features].values[len(X_tr):len(X_tr)+len(X_te)], y_te)
std_knn = KNeighborsClassifier(n_neighbors=5).fit(X_tr, y_tr)
std_acc = std_knn.score(X_te, y_te)
print(f"  KNN accuracy WITHOUT scaling: {raw_acc:.3f}")
print(f"  KNN accuracy WITH scaling:    {std_acc:.3f}  <- higher")
print(f"  capital_gain ranges 0-99999; age ranges 17-90")
print(f"  Without scaling, capital_gain dominates distance completely")


In [ ]:
# Visualize KNN decision boundary using 2 most informative features
# age vs education_num — both interpretable business variables

import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,
    'axes.spines.top':False,'axes.spines.right':False,'font.size':11
})

X_2d = scaler.transform(adult[['age','education_num']].values)
y_2d = adult['income_binary'].values

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
K_vals = [1, 5, 15, 51]

for ax, K in zip(axes, K_vals):
    knn = KNeighborsClassifier(n_neighbors=K)
    knn.fit(X_2d, y_2d)

    xx, yy = np.meshgrid(np.linspace(X_2d[:,0].min()-0.5, X_2d[:,0].max()+0.5, 200),
                         np.linspace(X_2d[:,1].min()-0.5, X_2d[:,1].max()+0.5, 200))
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.2, colors=['#1e5fa8','#e85d20'])
    ax.contour(xx, yy, Z, colors=['#1e5fa8','#e85d20'], linewidths=1)
    ax.scatter(X_2d[y_2d==0,0], X_2d[y_2d==0,1],
               color='#1e5fa8', alpha=0.15, s=6, label='<=50K')
    ax.scatter(X_2d[y_2d==1,0], X_2d[y_2d==1,1],
               color='#e85d20', alpha=0.15, s=6, label='>50K')

    cv_acc = cross_val_score(KNeighborsClassifier(n_neighbors=K),
                             X_2d, y_2d, cv=5).mean()
    ax.set_title(f'K = {K}\n5-fold CV acc = {cv_acc:.3f}', fontsize=10)
    ax.set_xlabel('Age (standardized)')
    if K == 1: ax.set_ylabel('Education yrs (standardized)')

plt.suptitle('KNN Decision Boundaries — Adult Income Dataset\n'
             'Blue = predict <=50K, Orange = predict >50K',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 K=1 memorizes every training point — jagged boundary, overfits")
print("   K=51 smooths everything — may miss fine-grained income patterns")
print("   The right K balances these extremes — use cross-validation to find it")


## 🔁 Part 2 — Choosing K with Cross-Validation

The right K balances the bias-variance tradeoff. We use 10-fold cross-validation to find the K that best generalizes to unseen data.

In [ ]:
# Choose K via cross-validation on the full feature set
K_range = range(1, 51)
cv_scores = []

for K in K_range:
    knn = KNeighborsClassifier(n_neighbors=K, metric='euclidean')
    scores = cross_val_score(knn, X_s, y, cv=10, scoring='accuracy', n_jobs=-1)
    cv_scores.append(scores.mean())

best_K = list(K_range)[cv_scores.index(max(cv_scores))]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(list(K_range), cv_scores, 'o-', color='#1e5fa8', lw=1.5, markersize=4)
ax.axvline(best_K, color='#e85d20', lw=2, ls='--',
           label=f'Best K={best_K}  (acc={max(cv_scores):.3f})')
ax.set_xlabel('K (number of neighbors)')
ax.set_ylabel('10-Fold CV Accuracy')
ax.set_title('Choosing K via Cross-Validation\nAdult Income Dataset — predict income >$50K')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\n📌 Optimal K = {best_K}")
print(f"   Training accuracy at K=1:  {KNeighborsClassifier(1).fit(X_tr,y_tr).score(X_tr,y_tr):.3f}  (memorizes data)")
print(f"   Test accuracy at K=1:      {KNeighborsClassifier(1).fit(X_tr,y_tr).score(X_te,y_te):.3f}  (generalizes poorly)")
print(f"   Test accuracy at K={best_K}: {KNeighborsClassifier(best_K).fit(X_tr,y_tr).score(X_te,y_te):.3f}  (better generalization)")

# Final model report
knn_best = KNeighborsClassifier(n_neighbors=best_K).fit(X_tr, y_tr)
y_pred = knn_best.predict(X_te)
print(f"\n=== Classification Report (K={best_K}) ===")
print(classification_report(y_te, y_pred, target_names=['<=50K', '>50K']))


## 📐 Part 3 — The Curse of Dimensionality

As dimensions increase, all points become approximately equidistant — the concept of 'nearest neighbor' loses meaning. Adding irrelevant features hurts KNN more than almost any other algorithm.

In [ ]:
# The Curse of Dimensionality — visualized with real data
# As we add more features, KNN performance can actually degrade

from sklearn.metrics import accuracy_score

feature_sets = {
    '2 features\n(age, educ)':
        ['age','education_num'],
    '3 features\n(+hours)':
        ['age','education_num','hours_per_week'],
    '5 features\n(all numeric)':
        ['age','education_num','hours_per_week','capital_gain','capital_loss'],
}

# Simulate high-dimensional noise features
accs = {}
np.random.seed(42)
base_X = adult[['age','education_num','hours_per_week','capital_gain','capital_loss']].values
base_y = adult['income_binary'].values

dim_results = []
for n_dims in [2, 5, 10, 20, 50, 100]:
    if n_dims <= 5:
        X_d = StandardScaler().fit_transform(base_X[:, :n_dims])
    else:
        noise = np.random.randn(len(base_X), n_dims - 5)
        X_d = StandardScaler().fit_transform(np.hstack([base_X, noise]))
    scores = cross_val_score(KNeighborsClassifier(n_neighbors=15),
                             X_d, base_y, cv=5)
    dim_results.append((n_dims, scores.mean()))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

dims, accs_d = zip(*dim_results)
axes[0].plot(dims, accs_d, 'o-', color='#e85d20', lw=2, markersize=7)
axes[0].axvline(5, color='#1e5fa8', lw=1.5, ls='--', alpha=0.6,
                label='Real features end')
axes[0].set_xlabel('Number of Features (dimensions)')
axes[0].set_ylabel('5-Fold CV Accuracy')
axes[0].set_title('Curse of Dimensionality — KNN on Adult Income\nAccuracy degrades as noise dimensions are added')
axes[0].legend()

# Distance concentration: in high dims all distances become similar
n_sample = 500
X_sample = StandardScaler().fit_transform(base_X[:n_sample])
dim_list = [2, 5, 10, 20, 50, 100]
std_ratio = []
for d in dim_list:
    if d <= 5:
        Xd = X_sample[:, :d]
    else:
        Xd = np.hstack([X_sample, np.random.randn(n_sample, d-5)])
    from sklearn.metrics import pairwise_distances
    dists = pairwise_distances(Xd[:100], Xd[100:200]).flatten()
    std_ratio.append(dists.std() / dists.mean())

axes[1].plot(dim_list, std_ratio, 's-', color='#1e5fa8', lw=2, markersize=7)
axes[1].set_xlabel('Number of Dimensions')
axes[1].set_ylabel('Std(distances) / Mean(distances)')
axes[1].set_title('Distance Concentration in High Dimensions\n(lower = all neighbors equidistant = KNN breaks down)')

plt.tight_layout()
plt.show()
print("\n📌 With real data: 5 meaningful features outperforms 100 noisy ones")
print("   The 'nearest neighbors' in 100D are barely closer than random points")


## 💼 Exercise

**Task 1:** Add binary features (`sex`, `workclass`) to the model and compare accuracy to numeric-only KNN.

**Task 2:** Compare distance metrics (`euclidean`, `manhattan`, `chebyshev`) at K=15 using 5-fold CV.

**Task 3:** Use `KNeighborsRegressor` to predict `hours_per_week` from other features. Compare RMSE to a linear regression baseline.

In [ ]:
# ── Exercise workspace ──────────────────────────────────────────────────────
# Task 1: Add the 'sex' and 'workclass' features as binary/dummy variables
#         Retrain KNN and compare accuracy to the numeric-only model.
#         Does adding categorical features help?
# YOUR CODE HERE

# Task 2: Try different distance metrics — 'euclidean', 'manhattan', 'chebyshev'
#         Which performs best on this dataset?
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

for metric in ['euclidean', 'manhattan', 'chebyshev']:
    knn = KNeighborsClassifier(n_neighbors=15, metric=metric)
    # YOUR CODE HERE — compute 5-fold CV accuracy for each metric
    pass

# Task 3: KNN for regression — predict hours_per_week from other features
from sklearn.neighbors import KNeighborsRegressor
# YOUR CODE HERE


In [ ]:
# @title 📝 Quiz — K-Nearest Neighbors
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** How does KNN make a classification prediction for a new point?
# @markdown **Q2:** Why must features be standardized before applying KNN?
# @markdown **Q3:** What is the curse of dimensionality and how does it affect KNN?
# @markdown **Q4:** What happens to bias and variance as K increases?
# @markdown **Q5:** On the Adult Income dataset, would KNN be your first-choice model? Why or why not?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_="K-Nearest Neighbors"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why — what did I get right or wrong")
    print("  3. Give the ideal complete answer so I know exactly what was expected")
    print("  4. If I was wrong or partial, tell me the specific concept to review")
    print("     and where it appears in the notebook (e.g. Part 2, the SHAP beeswarm plot)")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan: which questions to re-read and what to focus on")
    print()
    print("After grading, I will paste specific outputs, charts, or code blocks")
    print("from the notebook — please explain them in detail and answer follow-up questions.")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong. This is how real open-source projects work.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*